# Adversarial Evaluation Report

**System under test:** customer support ticket classifier (text in, structured JSON out).

This notebook reads previously generated results from `results/` and does not
call the model. To reproduce the results, run the evaluation runner first:

`python -m evals.runner --prompt v1`

In [ ]:
import json, pathlib, os
import pandas as pd
from IPython.display import Image, display

if pathlib.Path.cwd().name == 'report':
    os.chdir('..')

R = pathlib.Path('results')
VERSION = 'v1'

raw = json.loads((R / f'raw_results_{VERSION}.json').read_text(encoding='utf-8'))
results = raw['results']
m = json.loads((R / f'metrics_{VERSION}.json').read_text(encoding='utf-8'))
clusters = json.loads((R / f'failure_clusters_{VERSION}.json').read_text(encoding='utf-8'))

acc, hal, cal = m['accuracy'], m['hallucination'], m['calibration']
raw['meta']

## 1. Executive summary

Summarize the main results in 3–4 sentences. State the overall performance,
the strongest and weakest areas, and the most important failure discovered.
Do not hide poor results; the purpose of this report is to characterize the
system's weaknesses honestly.

In [ ]:
pd.Series({
    'category accuracy (all)':   acc['overall_category_accuracy'],
    'category accuracy (clean)': acc['normal_accuracy'],
    'category accuracy (adversarial)': acc['adversarial_accuracy'],
    'priority accuracy':         acc['overall_priority_accuracy'],
    'exact match (both fields)': acc['overall_exact_accuracy'],
    'hallucination rate':        hal['hallucination_rate'],
    'ECE':                       cal['ece'],
    'MCE':                       cal['mce'],
    'mean overconfidence':       cal['mean_overconfidence'],
    'test cases':                acc['n_total'],
}).to_frame('value')

## 2. Ground truth construction

The ground-truth dataset contains 52 manually reviewed cases. Each case has an
expected category, priority, and a written rationale explaining the labeling
decision. Cases are also tagged by difficulty (easy, arguable, or hard) to
distinguish straightforward examples from cases where category or priority
boundaries are intentionally ambiguous.

Priority labels follow the rules below:

| priority | rule |
|---|---|
| high | money lost, access blocked, account compromised, product unusable or unsafe, or a stated deadline within 48 hours |
| medium | a real problem or time-sensitive request, but service is only degraded or delayed and nothing is lost yet |
| low | information requests, feedback, and minor complaints that leave the customer unaffected |

**Trust and limitations:** document who labelled the cases, how ambiguous cases
were resolved, which cases required reconsideration, and which labels remain
contestable. The `difficulty` field identifies cases intentionally included to
test borderline judgement.

In [ ]:
gt = pd.DataFrame(json.loads(pathlib.Path('data/ground_truth.json').read_text(encoding='utf-8')))
gt['category'] = gt['expected'].apply(lambda e: e['category'])
gt['priority'] = gt['expected'].apply(lambda e: e['priority'])
display(pd.crosstab(gt['category'], gt['priority'], margins=True))
display(gt['difficulty'].value_counts().to_frame('cases'))

## 3. Adversarial methodology

Each adversarial family targets a specific hypothesized weakness:

| family | what it changes | weakness it targets |
|---|---|---|
| paraphrase | wording only | sensitivity to surface form |
| irrelevant_context | adds unrelated text containing vocabulary from other categories | distraction and keyword bias |
| contradiction | appends conflicting information | handling of conflicting evidence |
| negation | introduces a decoy category inside a negated clause | keyword matching without understanding scope |
| typos | introduces character-level noise | input brittleness and evidence extraction |
| ood | replaces the support request with an off-domain input | inappropriate confidence outside the task domain |

For derived adversarial cases, the expected label is inherited from the
human-verified base case. The generator never asks an LLM to determine the
ground-truth label.

Paraphrases are generated with an LLM, disclosed as such, and stored in
`data/adversarial.json` so they can be inspected and reproduced without
regenerating them.

In [ ]:
by_type = pd.DataFrame(acc['by_input_type']).T
by_type = by_type[['n', 'category_accuracy', 'priority_accuracy', 'mean_confidence']]
by_type.sort_values('category_accuracy')

## 4. Results by category and confusion matrix

In [ ]:
display(pd.DataFrame(acc['per_class']).T)
cm = pd.DataFrame(acc['confusion']).T
cm.index.name = 'gold'
cm.columns.name = 'predicted'
display(cm)

## 5. Hallucination detection

For this system, hallucination detection focuses on whether the model produces
unsupported evidence or output values that cannot be grounded in the input.

Task correctness and evidence grounding are evaluated separately:
- `category` and `priority` are checked against the ground-truth labels.
- `evidence_span` is checked against the original ticket.

The evidence span is classified as:
- `supported`: the span appears exactly in the input after normalization.
- `paraphrased_unverified`: the span is similar to source text but is not an
  exact match, so it is not treated as fully verified.
- `fabricated`: the span contains material that cannot be found in the input.

Only `fabricated` spans contribute to the reported hallucination rate.

**Limitations:** an evidence span can be copied correctly while still failing
to actually justify the predicted label. Exact or fuzzy matching can also
produce false positives or false negatives, particularly for numbers,
word-form changes, and long inputs.

In [ ]:
if not hal:
    print('no scored results: every case errored, check the error field in raw_results')
else:
    display(pd.Series(hal['span_verdicts']).to_frame('count'))
    display(pd.DataFrame(hal['by_input_type']).T.sort_values('rate', ascending=False))

    bad = [r for r in results if r.get('hallucination', {}).get('span_verdict') == 'fabricated'][:5]
    for r in bad:
        print(f"[{r['test_id']}] {r['input'][:90]}")
        print(f"   span: {r['predicted']['evidence_span'][:90]}")
        print(f"   missing words: {r['hallucination']['span_missing_words']}\n")

## 6. Confidence calibration

In [ ]:
b = pd.DataFrame(cal['buckets'])
display(b[b['n'] > 0][['lo', 'hi', 'n', 'mean_confidence', 'accuracy', 'gap']])
print('high-confidence slice (>= 0.9):', cal['high_confidence_slice'])
print('out-of-distribution slice:     ', cal['ood_slice'])
p = R / f'calibration_{VERSION}.png'
if p.exists():
    display(Image(filename=str(p)))

## 7. Failure mode clusters

Each cluster represents a recurring failure pattern identified from the
evaluation results. For every cluster, report:

1. the observed failure pattern,
2. a model-behaviour hypothesis explaining why it occurs,
3. representative examples, and
4. one concrete improvement that could address the failure.

Do not claim a root cause that is not supported by the observed results.

In [ ]:
display(pd.DataFrame([
    {'cluster': c['label'], 'failures': c['n_failures'],
     'share': c['share_of_failures'], 'mean_conf': c['mean_confidence']}
    for c in clusters['clusters']
]))

for c in clusters['clusters']:
    print('=' * 78)
    print(f"{c['label']}  —  {c['n_failures']} failures, mean confidence {c['mean_confidence']}")
    print('-' * 78)
    print('WHY:', c['hypothesis'])
    print('\nFIX:', c['proposed_fix'])
    print('\nTop confusions:', c['top_confusions'])
    print('\nExamples:')
    for e in c['examples']:
        print(f"  [{e['test_id']}] {e['expected']} -> {e['predicted']} (conf {e['confidence']})")
        print(f"      {e['input'][:110]}")
    print()

## 8. Regression test

v1 is the baseline prompt. v2 introduces targeted changes based on the failure
modes observed in v1.

The regression runner compares each test individually rather than relying only
on aggregate accuracy. This identifies cases that passed under v1 but fail
under v2, allowing prompt changes to be evaluated for both improvements and
regressions.

In [ ]:
p = R / 'regression_report.json'
if p.exists():
    reg = json.loads(p.read_text(encoding='utf-8'))
    display(pd.Series({k: v for k, v in reg.items() if not isinstance(v, (list, dict))}).to_frame('value'))
    print('regressions by input type:', reg['regressions_by_input_type'])
    if reg['regressions']:
        display(pd.DataFrame(reg['regressions'])[['test_id', 'input_type', 'expected', 'before', 'after', 'confidence_after']].head(15))
    else:
        print('no regressions: every test that passed on v1 still passes on v2')
else:
    print('run: python -m evals.runner --prompt v2  then  python -m evals.regression')

## 9. Limitations and next steps

Describe the main limitations of this evaluation honestly. At minimum, discuss
the dataset size, single-model evaluation, single-labeller ground truth,
chosen adversarial families, and any assumptions made by the hallucination and
calibration metrics.

Then identify the most valuable next experiment or improvement.